In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import gc
import time

start_time = time.time()

# --- 1. Strict Kaggle Paths ---
COMP_DIR = '/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage' 
TRAIN_PATH = f'{COMP_DIR}/train.parquet'
TEST_PATH = f'{COMP_DIR}/test.parquet'

def reduce_mem_usage(df):
    """Aggressive memory compression."""
    for col in df.columns:
        if df[col].dtype != object:
            if str(df[col].dtype)[:3] == 'int':
                df[col] = pd.to_numeric(df[col], downcast='integer')
            else:
                df[col] = pd.to_numeric(df[col], downcast='float')
    return df

# --- 2. Advanced Row-Wise Feature Engineering (100% Compliant) ---
def engineer_features(df):
    """
    Extracts row-wise momentum and volatility without crossing samples.
    This provides massive signal to tree models while avoiding leakage penalties.
    """
    print("Engineering row-wise quant features...")
    
    # 1. Identify Lag columns
    lag1_cols = [c for c in df.columns if '_LagT1' in c]
    lag2_cols = [c for c in df.columns if '_LagT2' in c]
    lag3_cols = [c for c in df.columns if '_LagT3' in c]
    
    # 2. Row-wise Volatility (Standard Deviation across lags for each row)
    # This measures how erratic the asset was during this specific snapshot
    df['row_lag1_std'] = df[lag1_cols].std(axis=1)
    df['row_lag2_std'] = df[lag2_cols].std(axis=1)
    
    # 3. Row-wise Macro Momentum (Mean movement across all features)
    df['row_lag1_mean'] = df[lag1_cols].mean(axis=1)
    
    # 4. Specific Feature Acceleration (Taking a subset to save RAM)
    # We calculate the momentum (Lag1 - Lag2) for the first 20 lag features
    for i in range(min(20, len(lag1_cols))):
        col1 = lag1_cols[i]
        col2 = col1.replace('_LagT1', '_LagT2')
        if col2 in df.columns:
            new_col_name = col1.replace('_LagT1', '_Momentum')
            df[new_col_name] = df[col1] - df[col2]
            
    return df

print("1. Loading datasets...")
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

print("2. Aligning base features...")
# Isolate TARGET and ID, find common base features
y_train = train['TARGET']
test_ids = test['ID']

base_features = [col for col in train.columns if col in test.columns and col != 'ID']
X_train = train[base_features]
X_test = test[base_features]

del train, test
gc.collect()

# --- Apply Feature Engineering ---
X_train = engineer_features(X_train)
X_test = engineer_features(X_test)

# Compress immediately to survive Kaggle CPU limits
X_train = reduce_mem_usage(X_train)
X_test = reduce_mem_usage(X_test)

print(f"Total predictive features generated: {X_train.shape[1]}")

# --- 3. Initializing Final Model ---
# Learning rate lowered slightly to account for the new complex features
best_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,  # Adjusted for new features
    'num_leaves': 106,
    'max_depth': 13,
    'feature_fraction': 0.83,
    'bagging_fraction': 0.97,
    'bagging_freq': 3,
    'min_child_samples': 84,
    'verbose': -1,
    'n_jobs': -1,  
    'random_state': 42
}

final_model = lgb.LGBMRegressor(**best_params, n_estimators=600)

print("4. Training Master Alpha Model (CPU)...")
final_model.fit(X_train, y_train)

del X_train, y_train
gc.collect()

print("5. Generating Predictions...")
predictions = final_model.predict(X_test)

print("6. Creating submission file...")
submission = pd.DataFrame({
    'ID': test_ids,
    'TARGET': predictions
})

submission.to_csv('submission.csv', index=False)

execution_time = (time.time() - start_time) / 60
print(f"SUCCESS! Advanced Pipeline completed perfectly in {execution_time:.2f} minutes.")
print("This notebook utilizes pure row-wise feature engineering and is 100% compliant with iRage rules.")

1. Loading datasets...
2. Aligning base features...
Engineering row-wise quant features...
Engineering row-wise quant features...
Total predictive features generated: 468
4. Training Master Alpha Model (CPU)...
5. Generating Predictions...
6. Creating submission file...
SUCCESS! Advanced Pipeline completed perfectly in 4.34 minutes.
This notebook utilizes pure row-wise feature engineering and is 100% compliant with iRage rules.
